# Example 2: Intermediate Globe (Coastline step & Hemisphere splitting)

This notebook demonstrates how to create a more advanced globe that combines standard topographic displacement with a sharp, physical step boundary at the coastlines (loaded from a shapefile). We then use `export_hemispheres()` to split the displaced model into capped top and bottom hemispheres, making them easy to print flat on a build plate.

## Step 1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    LineDisplacer,
    calculate_displacement_scale
)

## Step 2: Generate base sphere and apply topography

We generate an 8,000-point Fibonacci sphere and apply ETOPO grid elevation displacement.

In [ ]:
model_radius_mm = 40.0
topo_units = 'm'  # ETOPO elevation data is in meters

model = GlobeModel(n_points=8000, radius=model_radius_mm)

full_grid = GeographicGrid.from_netcdf(
    "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=full_grid.lats[::10],
    lons=full_grid.lons[::10],
    grid=full_grid.grid[::10, ::10]
)

scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=40.0, grid_units=topo_units)
model.outer.displace(GridDisplacer(grid_ds), scale=scale)

## Step 3: Apply coastline step boundary

We use a Natural Earth coastline shapefile to identify vertices close to the shorelines and displace them by an extra 0.8 mm. This creates a clean tactile ledge at the coast on the physical print.

In [ ]:
model.outer.displace(LineDisplacer(
    shapefile_path="../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,   # 0.8 mm step height
    width_degrees=0.5   # 0.5 degrees ribbon width
))
print("Applied coastline step displacement.")

## Step 4: Preview both hemispheres

In [ ]:
# Generate hemispheres for preview (not hollow for this simple example)
top_half, bottom_half = model.generate_hemispheres(hollow=False)

fig = plt.figure(figsize=(12, 6))

ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=pts_top[:, 2], cmap='viridis', s=2)
ax1.set_title("Top Hemisphere")

ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=pts_bot[:, 2], cmap='viridis', s=2)
ax2.set_title("Bottom Hemisphere")

plt.show()

## Step 5: Export STL files

We use `export_hemispheres()` to split and export in one call.

In [ ]:
model.export_hemispheres(
    "../outputs/example_2_top.stl",
    "../outputs/example_2_bottom.stl",
    hollow=False,
)
print("Hemispheres exported to outputs/")